# Day 28 of ML 30-Days Challenge

# Task

**Hyperparameter Tuning**: Learn to fine-tune models using GridSearchCV. Understand the difference between model parameters (learned from data) and hyperparameters (set before learning) and explore RandomizedSearchCV as a more efficient alternative for large search spaces.

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

###1. Create a 200-row dataset (binary classification) -> it takes too long for large data

In [13]:
X, y = make_classification(
    n_samples=200, n_features=20, n_informative=6, n_redundant=4, n_repeated=0,
    n_classes=2, n_clusters_per_class=2, class_sep=1.2, flip_y=0.02, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.25, random_state=42, stratify=y )

###2. Baseline model (simple RandomForest with common defaults)

In [14]:
baseline = RandomForestClassifier(
    n_estimators=100, max_depth=None, random_state=42, n_jobs=-1
)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
base_acc = accuracy_score(y_test, y_pred_base)

print("Baseline accuracy:", base_acc)
print("\nBaseline classification report:\n", classification_report(y_test, y_pred_base))

Baseline accuracy: 0.86

Baseline classification report:
               precision    recall  f1-score   support

           0       0.91      0.80      0.85        25
           1       0.82      0.92      0.87        25

    accuracy                           0.86        50
   macro avg       0.87      0.86      0.86        50
weighted avg       0.87      0.86      0.86        50



###3. RandomizedSearchCV for efficient initial sweep

In [15]:
param_dist = {
    "n_estimators": randint(50, 400),
    "max_depth": randint(2, 30),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
}
rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=40,  # adjust for speed vs coverage
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=0,
)
rs.fit(X_train, y_train)
print("\nRandomizedSearchCV best params:\n", rs.best_params_)
print("RandomizedSearchCV best CV score:", rs.best_score_)


RandomizedSearchCV best params:
 {'bootstrap': False, 'max_depth': 13, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 11, 'n_estimators': 237}
RandomizedSearchCV best CV score: 0.8533333333333335


###4. GridSearchCV to refine around the RS best region

In [16]:
best_rs = rs.best_params_
grid = {
    "n_estimators": [best_rs["n_estimators"] - 50, best_rs["n_estimators"], best_rs["n_estimators"] + 50]
        if best_rs["n_estimators"] >= 100 else
        [max(50, best_rs["n_estimators"] - 25), best_rs["n_estimators"], best_rs["n_estimators"] + 25],
    "max_depth": [max(2, best_rs["max_depth"] - 3), best_rs["max_depth"], best_rs["max_depth"] + 3],
    "min_samples_split": [max(2, best_rs["min_samples_split"] - 2), best_rs["min_samples_split"], best_rs["min_samples_split"] + 2],
    "min_samples_leaf": [max(1, best_rs["min_samples_leaf"] - 1), best_rs["min_samples_leaf"], best_rs["min_samples_leaf"] + 1],
    "max_features": [best_rs["max_features"]],
    "bootstrap": [best_rs["bootstrap"]],
}
gs = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
gs.fit(X_train, y_train)
print("\nGridSearchCV best params:\n", gs.best_params_)
print("GridSearchCV best CV score:", gs.best_score_)


GridSearchCV best params:
 {'bootstrap': False, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 11, 'n_estimators': 237}
GridSearchCV best CV score: 0.8533333333333335


###5. Final tuned model evaluation

In [17]:
tuned = gs.best_estimator_
y_pred_tuned = tuned.predict(X_test)
tuned_acc = accuracy_score(y_test, y_pred_tuned)

print("\nTuned accuracy:", tuned_acc)


Tuned accuracy: 0.9
